In [0]:
import os, sys, subprocess

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT = f"{REPO}/dbt"
DBT_LIB = "/tmp/dbtlib"
PROFILES = "/tmp/dbt_profiles"

# 1. Install pinned dbt into its own folder (keeps notebook packages untouched)
if not os.path.exists(f"{DBT_LIB}/dbt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--ignore-installed",
                    "--target", DBT_LIB, "dbt-core==1.12.5", "dbt-databricks==1.12.7"], check=True)

# 2. Connection settings (token from the secret scope, never in the repo)
os.makedirs(PROFILES, exist_ok=True)
with open(f"{PROFILES}/profiles.yml", "w") as f:
    f.write("""oil_shock:
  target: prod
  outputs:
    prod:
      type: databricks
      catalog: oil_shock
      schema: silver
      host: dbc-050dc881-f83e.cloud.databricks.com
      http_path: /sql/1.0/warehouses/42944d45ee999e1c
      token: "{{ env_var('DBT_DATABRICKS_TOKEN') }}"
      threads: 4
""")

env = dict(os.environ,
           PYTHONPATH=DBT_LIB, PYTHONNOUSERSITE="1",
           DBT_DATABRICKS_TOKEN=dbutils.secrets.get("oil-shock", "databricks-token"),
           DBT_TARGET_PATH="/tmp/dbt_target", DBT_LOG_PATH="/tmp/dbt_logs",
           DBT_PROFILES_DIR=PROFILES)

# 3. Build everything (seeds, models, tests)
r = subprocess.run([sys.executable, "-c", "from dbt.cli.main import cli; cli()",
                    "build", "--project-dir", DBT],
                   capture_output=True, text=True, env=env)
print(r.stdout[-8000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("dbt build failed — see output above")
print("dbt build OK")

In [0]:
# Brent 30-day volatility as a gold table (dbt model + build now + check)  (one cell, <1 min)
import time
t0 = time.time()
def log(m): print(f"[{time.time()-t0:5.0f}s] {m}", flush=True)

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
DBT_FILE = f"{REPO}/dbt/models/marts/fct_brent_volatility_daily.sql"

DBT_SQL = """with p as (
    select price_date, avg(price) as brent_price
    from {{ ref('fct_price_daily') }}
    where series_id = 'RBRTE' and price is not null
    group by price_date
),

r as (
    select
        price_date,
        brent_price,
        ln(brent_price / lag(brent_price) over (order by price_date)) as log_return
    from p
),

w as (
    select
        price_date,
        brent_price,
        log_return,
        stddev_samp(log_return) over (order by price_date rows between 29 preceding and current row) as sd_30,
        count(log_return)       over (order by price_date rows between 29 preceding and current row) as n_30
    from r
)

select
    price_date,
    brent_price,
    log_return,
    case when n_30 >= 20 then sd_30 * sqrt(252) end as volatility_30d,
    n_30 as returns_in_window
from w
"""

# 1) dbt model file (the 21:00 job's dbt_build makes it official)
with open(DBT_FILE, "w") as f:
    f.write(DBT_SQL)
log("dbt file written: fct_brent_volatility_daily.sql")

# 2) Build the gold table now, same SQL as dbt
COMPILED = DBT_SQL.replace("{{ ref('fct_price_daily') }}", "oil_shock.gold.fct_price_daily")
spark.sql("CREATE OR REPLACE TABLE oil_shock.gold.fct_brent_volatility_daily AS " + COMPILED)
log("gold.fct_brent_volatility_daily built")

# 3) Checks: should match the report (latest ~78%, 2025 avg ~30%, April peak ~110%)
t = spark.table("oil_shock.gold.fct_brent_volatility_daily")
dup = t.groupBy("price_date").count().where("count > 1").count()
log(f"rows {t.count():,} | duplicate dates {dup}")
spark.sql("""
  SELECT 'latest' AS what, max_by(price_date, price_date) AS d, round(max_by(volatility_30d, price_date)*100,1) AS vol_pct
  FROM oil_shock.gold.fct_brent_volatility_daily
  UNION ALL
  SELECT '2025 avg', NULL, round(avg(volatility_30d)*100,1)
  FROM oil_shock.gold.fct_brent_volatility_daily WHERE year(price_date) = 2025
  UNION ALL
  SELECT 'war peak', max_by(price_date, volatility_30d), round(max(volatility_30d)*100,1)
  FROM oil_shock.gold.fct_brent_volatility_daily WHERE price_date >= '2026-02-28'
""").show(truncate=False)

log("DONE - paste this output back to Claude")

In [0]:
# Brent since 1987 in today's dollars + major events (seed + dbt model + build now + check)  (one cell, <1 min)
import time, csv
t0 = time.time()
def log(m): print(f"[{time.time()-t0:5.0f}s] {m}", flush=True)

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/dbt"
SEED_FILE = f"{REPO}/seeds/ref_oil_history_events.csv"
MODEL_FILE = f"{REPO}/models/marts/fct_brent_history_monthly.sql"
YML_FILE = f"{REPO}/models/marts/_marts_history.yml"

# Event month = the month it happened (2008 = the record high; 2014 = OPEC's decision not to cut)
EVENTS = [
    ("1990-08-01", "1990: Iraq invades Kuwait"),
    ("2003-03-01", "2003: US invades Iraq"),
    ("2008-07-01", "2008: record high, then crash"),
    ("2014-11-01", "2014: OPEC keeps output up"),
    ("2020-04-01", "2020: COVID lockdowns"),
    ("2022-02-01", "2022: Russia invades Ukraine"),
    ("2026-02-01", "2026: war begins"),
]

MODEL_SQL = """with p as (
    select
        cast(date_trunc('month', price_date) as date) as month_start,
        avg(price)                                    as brent_nominal,
        count(*)                                      as trading_days
    from {{ ref('fct_price_daily') }}
    where series_id = 'RBRTE' and price is not null
    group by 1
),

c as (
    select obs_date as month_start, value as cpi
    from {{ ref('stg_fred__observations') }}
    where series_id = 'CPIAUCSL'
),

last_c as (
    select max_by(cpi, month_start) as cpi_last, max(month_start) as cpi_last_month
    from c
),

e as (
    select cast(event_month as date) as month_start, event_label
    from {{ ref('ref_oil_history_events') }}
)

select
    p.month_start,
    p.brent_nominal,
    p.trading_days,
    coalesce(c.cpi, l.cpi_last)                                  as cpi,
    p.brent_nominal * l.cpi_last / coalesce(c.cpi, l.cpi_last)   as brent_real,
    l.cpi_last_month                                             as real_dollars_as_of,
    e.event_label
from p
cross join last_c l
left join c on c.month_start = p.month_start
left join e on e.month_start = p.month_start
"""

YML = """version: 2

models:
  - name: fct_brent_history_monthly
    description: Brent physical spot (EIA) monthly average since 1987, nominal and in today's dollars (US CPI, FRED CPIAUCSL), with major events marked. Months after the latest CPI use the latest CPI.
    columns:
      - name: month_start
        tests: [unique, not_null]
      - name: brent_real
        tests: [not_null]
"""

# 1) Files for dbt (the 21:00 job's dbt_build loads the seed, builds and tests the model)
with open(SEED_FILE, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["event_month", "event_label"])
    w.writerows(EVENTS)
with open(MODEL_FILE, "w") as f:
    f.write(MODEL_SQL)
with open(YML_FILE, "w") as f:
    f.write(YML)
log("written: seeds/ref_oil_history_events.csv, marts/fct_brent_history_monthly.sql, marts/_marts_history.yml")

# 2) Build now on notebook compute, same SQL as dbt
spark.createDataFrame(EVENTS, "event_month string, event_label string") \
     .selectExpr("cast(event_month as date) as event_month", "event_label") \
     .write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable("oil_shock.silver.ref_oil_history_events")
COMPILED = (MODEL_SQL
    .replace("{{ ref('fct_price_daily') }}", "oil_shock.gold.fct_price_daily")
    .replace("{{ ref('stg_fred__observations') }}", "oil_shock.silver.stg_fred__observations")
    .replace("{{ ref('ref_oil_history_events') }}", "oil_shock.silver.ref_oil_history_events"))
spark.sql("CREATE OR REPLACE TABLE oil_shock.gold.fct_brent_history_monthly AS " + COMPILED)
log("silver.ref_oil_history_events + gold.fct_brent_history_monthly built")

# 3) Checks
t = spark.table("oil_shock.gold.fct_brent_history_monthly")
r = t.selectExpr("count(*) n", "min(month_start) a", "max(month_start) b", "max(real_dollars_as_of) c").first()
dup = t.groupBy("month_start").count().where("count > 1").count()
log(f"months {r.n} ({r.a} to {r.b}) | duplicate months {dup} | real dollars as of {r.c}")

print("\nEvents (monthly average, $/bbl):")
spark.sql("""SELECT month_start, event_label, round(brent_nominal,1) AS nominal, round(brent_real,1) AS today_dollars
             FROM oil_shock.gold.fct_brent_history_monthly WHERE event_label IS NOT NULL ORDER BY 1""").show(truncate=False)

print("Highest month in each episode, today's dollars:")
spark.sql("""
  SELECT ep, max_by(month_start, brent_real) AS peak_month, round(max(brent_nominal),1) AS peak_nominal, round(max(brent_real),1) AS peak_today_dollars
  FROM (SELECT *, CASE
          WHEN month_start BETWEEN '1990-07-01' AND '1991-03-01' THEN '1990'
          WHEN month_start BETWEEN '2002-10-01' AND '2003-12-01' THEN '2003'
          WHEN month_start BETWEEN '2007-06-01' AND '2008-12-01' THEN '2008'
          WHEN month_start BETWEEN '2022-01-01' AND '2022-12-01' THEN '2022'
          WHEN month_start >= '2026-01-01' THEN '2026' END AS ep
        FROM oil_shock.gold.fct_brent_history_monthly)
  WHERE ep IS NOT NULL GROUP BY ep ORDER BY ep""").show(truncate=False)

log("DONE - paste this output back to Claude")

In [0]:
spark.sql("""MERGE INTO oil_shock.gold.fct_brent_history_monthly h
             USING oil_shock.silver.ref_oil_history_events e
             ON h.month_start = e.event_month
             WHEN MATCHED THEN UPDATE SET h.event_label = e.event_label""")
spark.sql("SELECT month_start, event_label FROM oil_shock.gold.fct_brent_history_monthly WHERE event_label IS NOT NULL ORDER BY 1").show()